<a href="https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [45]:
# Inspect what data is already loaded

print("Variables available:")
print([x for x in globals().keys() if not x.startswith("_")])

# If your main dataframe is called df, this will show its structure
if "df" in globals():
    print("\nShape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nSample:")
    display(df.head())

Variables available:
['In', 'Out', 'get_ipython', 'exit', 'quit', 'pd', 'np', 'Path', 'DATA_PATH', 'p', 'df', 'i', 'col', 'staleness_check', 'volume_check', 'stale_band', 'visible', 'queue', 'top20_ids', 'review_features', 'top20_review', 'weak_picks', 'score_inputs', 'forbidden_inputs', 'precision_at_k', 'labels', 'scores', 'base_rate', 'k']

Shape: (30000, 47)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,staleness_bucket,volume_bucket
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,5.88,4.55,0.0,good,striking,down,-41.4,1,<=20d,"(3615.25, 517715.0]"
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,21-60d,"(3615.25, 517715.0]"
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,<=20d,"(3615.25, 517715.0]"
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,1.28,3.45,0.0,good,page_1,stable,-13.8,0,21-60d,"(3615.25, 517715.0]"
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,<=20d,"(3615.25, 517715.0]"


In [46]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("\nPossible data files:")
for p in Path(".").rglob("content_refresh_anonymized.csv"):
    print(p)

Current directory: /content

Possible data files:
flyrank_work/data/raw/content_refresh_anonymized.csv


In [47]:
!git clone https://github.com/ishpree1t7/flyrank_work

fatal: destination path 'flyrank_work' already exists and is not an empty directory.


In [48]:
from pathlib import Path

print("Repo folders:")
print(list(Path(".").iterdir()))

print("\nLooking for dataset...")
for p in Path(".").rglob("content_refresh_anonymized.csv"):
    print(p)

Repo folders:
[PosixPath('.config'), PosixPath('flyrank_work'), PosixPath('sample_data')]

Looking for dataset...
flyrank_work/data/raw/content_refresh_anonymized.csv


In [49]:
import pandas as pd

DATA_PATH = "flyrank_work/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [50]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [51]:
print("Shape:", df.shape)
print("\nALL COLUMNS:")
for i, col in enumerate(df.columns):
    print(i, repr(col))

Shape: (30000, 44)

ALL COLUMNS:
0 'content_id'
1 'client_id'
2 'search_volume'
3 'competition'
4 'competition_level'
5 'cpc'
6 'content_type'
7 'main_intent'
8 'word_count'
9 'char_count'
10 'provider_used'
11 'model_used'
12 'impressions_90d'
13 'clicks_90d'
14 'pageviews_90d'
15 'sessions_90d'
16 'users_90d'
17 'engaged_sessions_90d'
18 'ai_sessions_90d'
19 'scroll_events_90d'
20 'days_with_impressions'
21 'days_with_sessions'
22 'impressions_last_30d'
23 'clicks_last_30d'
24 'sessions_last_30d'
25 'impressions_prev_30d'
26 'clicks_prev_30d'
27 'sessions_prev_30d'
28 'content_age_days'
29 'age_tier'
30 'age_tier_order'
31 'days_since_last_update'
32 'freshness_tier'
33 'word_count_tier'
34 'char_count_tier'
35 'ctr'
36 'avg_position'
37 'engagement_rate'
38 'scroll_rate'
39 'ai_traffic_pct'
40 'impression_tier'
41 'position_tier'
42 'trend_direction'
43 'trend_pct'


In [52]:
print("\nColumns containing 'declin', 'trend', 'label', or 'click':")

for col in df.columns:
    if any(x in col.lower() for x in ["declin", "trend", "label", "click"]):
        print(repr(col))


Columns containing 'declin', 'trend', 'label', or 'click':
'clicks_90d'
'clicks_last_30d'
'clicks_prev_30d'
'trend_direction'
'trend_pct'


In [53]:
print("trend_direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\ntrend_pct summary:")
print(df["trend_pct"].describe())

trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


In [54]:
print("days_since_last_update:")
print(df["days_since_last_update"].describe())

print("\nimpressions_90d:")
print(df["impressions_90d"].describe())

days_since_last_update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

impressions_90d:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64


In [55]:
print("\nMissing values:")
print(
    df[
        ["days_since_last_update", "impressions_90d",
         "trend_direction", "trend_pct"]
    ].isna().sum()
)


Missing values:
days_since_last_update       0
impressions_90d              0
trend_direction              0
trend_pct                 3388
dtype: int64


In [56]:
from pathlib import Path

print("Files in the repo containing 'label' or 'target':")

for p in Path("flyrank_work").rglob("*"):
    if p.is_file() and any(x in p.name.lower() for x in ["label", "target", "eval", "metric"]):
        print(p)

Files in the repo containing 'label' or 'target':
flyrank_work/scripts/04_evaluate_and_export.py


In [57]:
!grep -R "is_declining_label" flyrank_work --exclude="*.csv" --exclude="*.parquet" 2>/dev/null | head -30

flyrank_work/notebooks/02_your_first_readable_model.ipynb:        "df[\"is_declining_label\"] = df[\"trend_direction\"].str.lower().eq(\"down\").astype(int)\n",
flyrank_work/notebooks/02_your_first_readable_model.ipynb:        "print(df.shape[0], \"pages |  declining rate:\", round(df[\"is_declining_label\"].mean(), 3))"
flyrank_work/notebooks/02_your_first_readable_model.ipynb:        "y = df[\"is_declining_label\"].values\n",
flyrank_work/outputs/model_report.md:- Target: `is_declining_label`
flyrank_work/GUIDE.md:**The label:** `is_declining_label = (trend_direction == "down")`. Because the label is
flyrank_work/docs/ml-intern-dataset-and-lane-guide.md:is_declining_label = trend_direction == "down"
flyrank_work/docs/data-dictionary.md:   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
flyrank_work/docs/data-dictionary.md:| `is_declining_label` | **The target.** 1 when `trend_direction == "down"` (16,262 rows = 54.2%), else 0 |
flyrank_work/sc

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule

Prioritize content that is both stale and has meaningful observed visibility. The baseline score combines staleness with impressions so that older content with more existing visibility is ranked earlier for review. This is a transparent decision-support rule, not a prediction model.

### Reason codes

- `stale_and_visible` — content is stale and has meaningful observed impressions.
- `stale_only` — content is stale but has lower observed visibility.
- `visible_only` — content has meaningful visibility but is not stale.
- `no_priority_signal` — neither condition is met.

The evaluation label `is_declining_label` is derived from `trend_direction == "down"` and is used only for audit/evaluation, never as a rule input.

In [58]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
DATA_PATH = Path("flyrank_work/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

# Evaluation-only target.
# NEVER use this column in the score.
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Shape:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Shape: (30000, 45)
Declining rate: 0.542


In [59]:
# Signal check 1: staleness

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 20, 60, 104, np.inf],
    labels=["<=20d", "21-60d", "61-104d", "105d+"],
    include_lowest=True
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean"),
          median_days=("days_since_last_update", "median")
      )
      .reset_index()
)

staleness_check["declining_rate"] = (
    staleness_check["declining_rate"] * 100
).round(1)

display(staleness_check)

,staleness_bucket,n,declining_rate,median_days
0,<=20d,15866,53.9,20.0
1,21-60d,4742,42.1,22.0
2,61-104d,9074,61.1,104.0
3,105d+,318,54.7,183.0


### Signal verdict — days_since_last_update: MIXED

Observed decline rates are not monotonic across staleness buckets: 53.9% for <=20 days, 42.1% for 21–60 days, 61.1% for 61–104 days, and 54.7% for 105+ days. The 61–104 day bucket is directionally higher, but the pattern does not support a simple "older always means more likely to decline" rule. I therefore treat staleness as a review signal rather than a monotonic risk measure.

In [60]:
# Signal check 2: observed visibility / volume

df["volume_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

volume_check["declining_rate"] = (
    volume_check["declining_rate"] * 100
).round(1)

display(volume_check)

,volume_bucket,n,declining_rate,median_impressions
0,"(0.999, 81.0]",7503,37.6,10.0
1,"(81.0, 731.0]",7499,60.5,300.0
2,"(731.0, 3615.25]",7498,62.6,1616.0
3,"(3615.25, 517715.0]",7500,56.2,9579.5


In [61]:
score_inputs = {
    "days_since_last_update",
    "impressions_90d"
}

forbidden_inputs = {
    "is_declining_label",
    "trend_direction",
    "trend_pct"
}

print("Score inputs:", sorted(score_inputs))
print("Forbidden inputs:", sorted(forbidden_inputs))

assert score_inputs.isdisjoint(forbidden_inputs)

print("\nLeakage check: PASSED")

Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden inputs: ['is_declining_label', 'trend_direction', 'trend_pct']

Leakage check: PASSED


### Baseline evaluation

The observed decline base rate is 54.2%. The rule achieves 60.0% precision@10, 45.0% precision@20, and 44.0% precision@50. The top-10 ranking is modestly above the base rate, while precision falls below the base rate at larger K. I treat this as directional decision-support evidence rather than proof that the rule identifies declining content reliably.

### Signal verdict — impressions_90d: MIXED

Observed decline rates increase from 37.6% in the lowest-impression bucket to 60.5% and 62.6% in the middle buckets, then fall to 56.2% in the highest-impression bucket. The signal is therefore directional but not monotonic. I treat impressions as a useful visibility/opportunity signal rather than evidence that higher volume always means higher decline risk.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [62]:
 #Transparent hand-written baseline

stale_band = (
    df["days_since_last_update"].between(61, 104)
)

visible = (
    df["impressions_90d"] >= 81
)

df["score"] = (
    stale_band.astype(int) * visible.astype(int) * df["impressions_90d"]
)

df["reason_code"] = np.select(
    [
        stale_band & visible,
        stale_band & ~visible,
        ~stale_band & visible
    ],
    [
        "stale_band_and_visible",
        "stale_band_only",
        "visible_only"
    ],
    default="no_priority_signal"
)

df["action"] = np.where(
    df["score"] > 0,
    "REVIEW",
    "NO_ACTION"
)

queue = (
    df[
        [
            "content_id",
            "client_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d"
        ]
    ]
    .sort_values(
        ["score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

display(queue.head(20))

,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_5fe46e04994d,client_4e07408562,517715,stale_band_and_visible,REVIEW,104,517715
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,stale_band_and_visible,REVIEW,104,443434
2,3,content_2c2606c5d176,client_19581e27de,347399,stale_band_and_visible,REVIEW,104,347399
3,4,content_cb112fce36be,client_19581e27de,309910,stale_band_and_visible,REVIEW,104,309910
4,5,content_9532f197bbc8,client_4e07408562,309192,stale_band_and_visible,REVIEW,104,309192
5,6,content_36ff89c8214e,client_19581e27de,295097,stale_band_and_visible,REVIEW,104,295097
6,7,content_b28d1efd668f,client_6208ef0f77,286608,stale_band_and_visible,REVIEW,104,286608
7,8,content_813e88069237,client_6208ef0f77,233561,stale_band_and_visible,REVIEW,104,233561
8,9,content_c21024970297,client_19581e27de,211366,stale_band_and_visible,REVIEW,104,211366
9,10,content_c8e9d6ab9013,client_19581e27de,208678,stale_band_and_visible,REVIEW,104,208678


In [63]:
OUTPUT_PATH = Path(
    "flyrank_work/work/outputs/baseline_action_score.csv"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Wrote:", OUTPUT_PATH)
print("Rows:", len(queue))

Wrote: flyrank_work/work/outputs/baseline_action_score.csv
Rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [64]:
top20_ids = queue.head(20)["content_id"]

review_features = df[
    df["content_id"].isin(top20_ids)
][[
    "content_id",
    "content_type",
    "main_intent",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "freshness_tier",
    "word_count"
]].copy()

review_features = (
    review_features
    .set_index("content_id")
    .loc[top20_ids]
    .reset_index()
)

display(review_features)

,content_id,content_type,main_intent,search_volume,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,freshness_tier,word_count
0,content_5fe46e04994d,keyword article,informational,1900.0,517715,741,0.14,4.2,537,104,91-180,NaN
1,content_2dba2b1f9536,keyword article,informational,0.0,443434,910,0.21,27.9,299,104,91-180,7676.0
2,content_2c2606c5d176,keyword article,commercial,590.0,347399,1854,0.53,4.2,362,104,91-180,NaN
3,content_cb112fce36be,keyword article,transactional,70.0,309910,492,0.16,5.6,126,104,91-180,2761.0
4,content_9532f197bbc8,keyword article,informational,10.0,309192,2689,0.87,2.0,445,104,91-180,NaN
5,content_36ff89c8214e,keyword article,informational,0.0,295097,154,0.05,7.3,144,104,91-180,NaN
6,content_b28d1efd668f,keyword article,transactional,0.0,286608,169,0.06,26.2,153,104,91-180,6901.0
7,content_813e88069237,keyword article,commercial,0.0,233561,129,0.06,26.2,153,104,91-180,4610.0
8,content_c21024970297,keyword article,commercial,110.0,211366,870,0.41,5.1,126,104,91-180,2874.0
9,content_c8e9d6ab9013,keyword article,informational,20.0,208678,0,0.00,9.7,362,104,91-180,NaN


In [65]:
top20_review = review_features[[
    "content_id",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "search_volume"
]].copy()

top20_review["action"] = "REVIEW"
top20_review["reason_code"] = "stale_band_and_visible"

top20_review["confidence_note"] = (
    "High visibility and 61-104 day staleness band support review."
)

top20_review["what_would_make_it_wrong"] = (
    "The page may already be performing well, so staleness and visibility "
    "alone may not justify a refresh."
)

display(top20_review)

,content_id,impressions_90d,days_since_last_update,avg_position,ctr,search_volume,action,reason_code,confidence_note,what_would_make_it_wrong
0,content_5fe46e04994d,517715,104,4.2,0.14,1900.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
1,content_2dba2b1f9536,443434,104,27.9,0.21,0.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
2,content_2c2606c5d176,347399,104,4.2,0.53,590.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
3,content_cb112fce36be,309910,104,5.6,0.16,70.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
4,content_9532f197bbc8,309192,104,2.0,0.87,10.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
5,content_36ff89c8214e,295097,104,7.3,0.05,0.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
6,content_b28d1efd668f,286608,104,26.2,0.06,0.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
7,content_813e88069237,233561,104,26.2,0.06,0.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
8,content_c21024970297,211366,104,5.1,0.41,110.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."
9,content_c8e9d6ab9013,208678,104,9.7,0.00,20.0,REVIEW,stale_band_and_visible,High visibility and 61-104 day staleness band ...,"The page may already be performing well, so st..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [66]:
weak_picks = queue[
    queue["reason_code"].isin(
        ["stale_only", "visible_only"]
    )
].head(10)

display(weak_picks)

,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
8117,8118,content_aaef01a50def,client_19581e27de,0,visible_only,NO_ACTION,22,517109
8118,8119,content_8c19996aa890,client_4e07408562,0,visible_only,NO_ACTION,20,509252
8119,8120,content_2cb567c3c89b,client_6208ef0f77,0,visible_only,NO_ACTION,48,497727
8120,8121,content_4c36c775b818,client_4e07408562,0,visible_only,NO_ACTION,20,463103
8121,8122,content_1a9e894be2e2,client_19581e27de,0,visible_only,NO_ACTION,22,416180
8122,8123,content_db5989a78dd3,client_4e07408562,0,visible_only,NO_ACTION,20,345111
8123,8124,content_44e481c8f55b,client_19581e27de,0,visible_only,NO_ACTION,20,312694
8124,8125,content_8e7ba84a972b,client_7f2253d7e2,0,visible_only,NO_ACTION,20,288426
8125,8126,content_89e84d699e9e,client_349c41201b,0,visible_only,NO_ACTION,20,275226
8126,8127,content_8451fc6f034d,client_d029fa3a95,0,visible_only,NO_ACTION,20,272144


In [67]:
score_inputs = {
    "days_since_last_update",
    "impressions_90d"
}

forbidden_inputs = {
    "is_declining_label",
    "trend_direction",
    "trend_pct"
}

print("Score inputs:", sorted(score_inputs))
print("Forbidden inputs:", sorted(forbidden_inputs))

assert score_inputs.isdisjoint(forbidden_inputs)

print("\nLeakage check: PASSED")

Score inputs: ['days_since_last_update', 'impressions_90d']
Forbidden inputs: ['is_declining_label', 'trend_direction', 'trend_pct']

Leakage check: PASSED


In [68]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


labels = df["is_declining_label"].values
scores = df["score"].values

base_rate = labels.mean()

print("Base decline rate:", round(base_rate, 3))

for k in [10, 20, 50]:
    print(
        f"Precision@{k}: "
        f"{precision_at_k(scores, labels, k):.3f}"
    )

Base decline rate: 0.542
Precision@10: 0.600
Precision@20: 0.450
Precision@50: 0.440


### Baseline evaluation

The observed decline base rate is 54.2%. The rule achieves 60.0% precision@10, 45.0% precision@20, and 44.0% precision@50. The top-10 ranking is modestly above the base rate, while precision falls below the base rate at larger K. I treat this as directional decision-support evidence rather than proof that the rule identifies declining content reliably.

## Self-check

- [x] Every section is filled with markdown reasoning and executable code.
- [x] The notebook runs from top to bottom without errors.
- [x] Two signal checks have visible bucket tables with `n` values.
- [x] `days_since_last_update` verdict: MIXED.
- [x] `impressions_90d` verdict: MIXED.
- [x] The rule uses a score, one reason code, and an action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] The Top-20 review includes action, reason code, confidence note, and what would make each pick wrong.
- [x] Weak picks were inspected.
- [x] Precision@10, Precision@20, and Precision@50 are reported with the base rate.
- [x] The score uses only `days_since_last_update` and `impressions_90d`.
- [x] No `is_declining_label`, `trend_direction`, or `trend_pct` is used as a scoring input.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language such as observed, directional, and decision-support.